# 05 · A visual gallery of core verbs

Use this notebook as a hackathon menu. It applies reducers, transforms, a
generic function, and a shape-changing verb to the same cube. Every panel is a
different question expressed with the same grammar.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from cubedynamics import pipe, verbs as v

rng = np.random.default_rng(8)
time = pd.date_range("2023-01-01", periods=24, freq="MS")
y = np.arange(5)
x = np.arange(6)
trend = np.linspace(0, 2, time.size)[:, None, None]
season = 2 * np.sin(2 * np.pi * np.arange(time.size)[:, None, None] / 12)
landscape = np.linspace(-1, 1, y.size)[None, :, None] + np.linspace(0, 1, x.size)[None, None, :]
cube = xr.DataArray(
    10 + trend + season + landscape + rng.normal(0, 0.3, (24, 5, 6)),
    dims=("time", "y", "x"),
    coords={"time": time, "y": y, "x": x},
    name="signal",
)

mean_map = (pipe(cube) | v.mean(dim="time", keep_dim=False)).unwrap()
variance_map = (pipe(cube) | v.variance(dim="time", keep_dim=False)).unwrap()
anomaly = (pipe(cube) | v.anomaly(dim="time")).unwrap()
zscore = (pipe(cube) | v.zscore(dim="time")).unwrap()
clipped = (pipe(zscore) | v.apply(lambda value: value.clip(min=-1, max=1))).unwrap()
flat = (pipe(anomaly) | v.flatten_space(new_dim="pixel")).unwrap()

assert flat.dims == ("time", "pixel")
fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
mean_map.plot(ax=axes[0, 0], cmap="viridis")
axes[0, 0].set_title("v.mean: typical spatial pattern")
variance_map.plot(ax=axes[0, 1], cmap="magma")
axes[0, 1].set_title("v.variance: variable locations")
anomaly.isel(time=-1).plot(ax=axes[0, 2], cmap="RdBu_r", center=0)
axes[0, 2].set_title("v.anomaly: departure from normal")
zscore.sel(y=2, x=3).plot(ax=axes[1, 0], color="#3f6f72")
axes[1, 0].axhline(0, color="0.45", linewidth=0.8)
axes[1, 0].set_title("v.zscore: comparable units")
clipped.isel(time=-1).plot(ax=axes[1, 1], cmap="RdBu_r", vmin=-1, vmax=1)
axes[1, 1].set_title("v.apply: project function")
axes[1, 2].imshow(flat.values.T, aspect="auto", cmap="RdBu_r")
axes[1, 2].set(title="v.flatten_space: time × pixel", xlabel="time index", ylabel="pixel")
plt.show()